# House Prices - EDA

Цель этого ноутбука — исследовать данные о ценах на жильё в Айове (Ames, Iowa), выявить закономерности и подготовить почву для построения моделей машинного обучения.

Мы проанализируем:
- структуру данных и пропуски
- распределение целевой переменной `SalePrice`
- взаимосвязи признаков с ценой
- корреляции между признаками
- выбросы и аномалии

Выводы будут использованы в `feature_engineering.py` и при выборе моделей.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('fivethirtyeight')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

In [ ]:
train.info()

In [ ]:
train.isnull().sum().sort_values(ascending=False).head(20)

### Пропуски
- **PoolQC** (1453 пропуска) — почти все пропущены, удалим
- **MiscFeature** (1406) — удалим
- **Alley** (1369) — удалим
- **Fence** (1179) — удалим
- **FireplaceQu** (690) — заполним `'None'`
- **LotFrontage** (259) — заполним медианой по соседству
- **Garage** (несколько) — заполним `'None'`
- **Bsmt** (несколько) — заполним `'None'`

В нашем пайплайне (`preprocessor.py`) мы удаляем признаки с >50% пропусков и заполняем остальные.

## Целевая переменная SalePrice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(train['SalePrice'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Распределение SalePrice (оригинал)')
axes[0].set_xlabel('Price')

sns.histplot(np.log1p(train['SalePrice']), bins=50, kde=True, ax=axes[1])
axes[1].set_title('Распределение log1p(SalePrice)')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.show()

# Вывод: логарифмическое преобразование делает распределение более нормальным,
# что полезно для регрессионных моделей.

## Анализ признаков

### Категориальные признаки vs SalePrice

In [ ]:
# Выберем категориальные признаки с небольшим числом уникальных значений
cat_cols = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 
            'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
            'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
            'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
            'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
            'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
            'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
            'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType',
            'SaleCondition']

# Посмотрим на топ-5 категорий по влиянию на цену
top_cat = ['Neighborhood', 'OverallQual', 'CentralAir', 'KitchenQual', 'ExterQual']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(top_cat):
    row, col_idx = divmod(i, 3)
    if col in train.columns:
        sns.boxplot(x=col, y='SalePrice', data=train, ax=axes[row, col_idx])
        axes[row, col_idx].set_title(f'SalePrice vs {col}')
        axes[row, col_idx].set_xticklabels(axes[row, col_idx].get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

# Вывод: Neighborhood, OverallQual, KitchenQual, ExterQual, CentralAir сильно влияют на цену.

### Числовые признаки vs SalePrice

In [ ]:
num_cols = ['LotArea', 'YearBuilt', 'YearRemodAdd', 'TotalBsmtSF', '1stFlrSF',
            '2ndFlrSF', 'GrLivArea', 'FullBath', 'HalfBath', 'BedroomAbvGr',
            'KitchenAbvGr', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF',
            'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea', 'MiscVal']

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for i, col in enumerate(num_cols):
    row, col_idx = divmod(i, 5)
    if col in train.columns:
        sns.scatterplot(x=col, y='SalePrice', data=train, ax=axes[row, col_idx], alpha=0.5)
        axes[row, col_idx].set_title(f'{col} vs SalePrice')
plt.tight_layout()
plt.show()

# Вывод:
# - GrLivArea, TotalBsmtSF, GarageArea имеют сильную положительную корреляцию с ценой
# - YearBuilt, YearRemodAdd — более новые дома дороже
# - LotArea — слабая корреляция, есть выбросы

## Корреляционная матрица

In [ ]:
# Берём числовые признаки с высоким влиянием
important_num = ['SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
                 'TotalBsmtSF', '1stFlrSF', 'FullBath', 'YearBuilt', 'YearRemodAdd',
                 'Fireplaces', 'LotArea']

plt.figure(figsize=(12, 10))
sns.heatmap(train[important_num].corr(), annot=True, cmap='RdYlGn', linewidths=0.5, fmt='.2f')
plt.title('Корреляция важных числовых признаков с SalePrice')
plt.show()

# Вывод:
# - OverallQual (0.79), GrLivArea (0.71), GarageCars (0.64), TotalBsmtSF (0.61) — сильные корреляции
# - LotArea (0.26) — слабая корреляция
# - YearBuilt и YearRemodAdd коррелируют между собой (0.56)

## Выбросы

In [ ]:
# График зависимости жилой площади от цены
plt.figure(figsize=(10, 6))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=train, alpha=0.6)
plt.title('GrLivArea vs SalePrice (выбросы)')
plt.show()

# Вывод: несколько точек с большой площадью (>4000) и низкой ценой (<200000) — возможно, выбросы
# В пайплайне мы не удаляем их, т.к. модели устойчивы к выбросам (бустинги, лес)
# Но можно добавить обработку выбросов в feature_engineering

## Feature Engineering (как в коде)

In [ ]:
# Пример создания новых признаков, как в feature_engineering.py
train_copy = train.copy()

# Общая площадь
train_copy['TotalSF'] = train_copy['TotalBsmtSF'] + train_copy['1stFlrSF'] + train_copy['2ndFlrSF']

# Общее качество
train_copy['OverallQuality'] = train_copy['OverallQual'] * train_copy['OverallCond']

# Возраст дома
train_copy['HouseAge'] = train_copy['YrSold'] - train_copy['YearBuilt']

# Ремонт
train_copy['YearsSinceRemod'] = train_copy['YearRemodAdd'] - train_copy['YearBuilt']

# Санузлы
train_copy['TotalBath'] = (train_copy['FullBath'] + 0.5*train_copy['HalfBath'] + 
                          train_copy['BsmtFullBath'] + 0.5*train_copy['BsmtHalfBath'])

print("Новые признаки добавлены:", train_copy.columns[-5:].tolist())

# Проверим корреляцию новых признаков с SalePrice
new_cols = ['TotalSF', 'OverallQuality', 'HouseAge', 'YearsSinceRemod', 'TotalBath']
plt.figure(figsize=(10, 6))
sns.heatmap(train_copy[new_cols + ['SalePrice']].corr(), annot=True, cmap='RdYlGn', fmt='.2f')
plt.title('Корреляция новых признаков с SalePrice')
plt.show()

# Вывод:
# - TotalSF и TotalBath хорошо коррелируют с ценой (0.73 и 0.72)
# - OverallQuality (0.65) тоже неплохо
# - HouseAge и YearsSinceRemod — отрицательная корреляция (старые дома дешевле)

## Итоги EDA

1. **Целевая переменная** — логарифмическое преобразование улучшает распределение.
2. **Сильные признаки:**
   - `OverallQual` (качество дома) — корреляция 0.79
   - `GrLivArea` (жилая площадь) — 0.71
   - `GarageCars` (машиноместа) — 0.64
   - `TotalBsmtSF` (подвал) — 0.61
3. **Категориальные признаки:** `Neighborhood`, `KitchenQual`, `ExterQual` сильно влияют на цену.
4. **Выбросы:** есть точки с большой площадью и низкой ценой — их можно не удалять, т.к. модели устойчивы.
5. **Новые признаки** (TotalSF, TotalBath, OverallQuality, HouseAge) улучшают модель.
6. **Признаки с >50% пропусков** (PoolQC, MiscFeature, Alley, Fence) — удаляем.

Эти выводы полностью соответствуют нашему пайплайну в `feature_engineering.py` и `preprocessor.py`.